# Basic Guardrails - LangGraph Workflow

This notebook demonstrates a simple, linear LangGraph pipeline representing the **Basic Input Validation Sandbox** protected by 3 sequential local validator nodes:

1. **Keyword Blocker**: Scans input for restricted keywords (`admin`, `root`, `sudo`, `hack`, `override`).
2. **Input Constraints**: Enforces string length boundaries (<100 characters) and blocks brackets (`[`, `]`, `<`, `>`).
3. **Sensitive Leak Blocker**: Scans for SSNs and Credit Cards using regex, redacting them if found.

---

### 1. Imports and Definitions

In [ ]:
import re
from typing import List, TypedDict
from langgraph.graph import StateGraph, START, END

class BasicState(TypedDict):
    text: str
    redacted_text: str
    keyword_passed: bool
    keyword_message: str
    constraint_passed: bool
    constraint_message: str
    leak_passed: bool
    leak_message: str
    overall_passed: bool
    logs: List[str]

### 2. Define Node Functions

In [ ]:
def check_keywords(state: BasicState) -> BasicState:
    text = state['text'].lower()
    banned = ['admin', 'root', 'sudo', 'hack', 'override']
    triggered = [w for w in banned if w in text]
    
    if triggered:
        state['keyword_passed'] = False
        state['keyword_message'] = f"Input contains prohibited keyword(s): {', '.join(triggered)}"
        state['logs'].append("[check_keywords]: Flagged prohibited keyword usage.")
    else:
        state['keyword_passed'] = True
        state['keyword_message'] = "Pass"
        state['logs'].append("[check_keywords]: Verified keyword safety.")
    return state

def check_constraints(state: BasicState) -> BasicState:
    text = state['text']
    
    # Check length constraint
    if len(text) > 100:
        state['constraint_passed'] = False
        state['constraint_message'] = f"Input length ({len(text)} characters) exceeds the 100 character maximum limit."
        state['logs'].append("[check_constraints]: Flagged length restriction violation.")
        return state
        
    # Check bracket injection constraint
    brackets = ['<', '>', '[', ']']
    found = [b for b in brackets if b in text]
    if found:
        state['constraint_passed'] = False
        state['constraint_message'] = f"Input contains restricted special bracket characters: {', '.join(found)}"
        state['logs'].append("[check_constraints]: Flagged injection bracket usage.")
    else:
        state['constraint_passed'] = True
        state['constraint_message'] = "Pass"
        state['logs'].append("[check_constraints]: Verified structural constraints.")
    return state

def check_leaks(state: BasicState) -> BasicState:
    text = state['text']
    
    # SSN Pattern
    ssn_pattern = r'\b\d{3}-\d{2}-\d{4}\b'
    # CC Pattern
    cc_pattern = r'\b(?:\d[ -]?){13,16}\b'
    
    has_ssn = bool(re.search(ssn_pattern, text))
    has_cc = bool(re.search(cc_pattern, text))
    
    redacted = text
    redacted = re.sub(ssn_pattern, "[SSN REDACTED]", redacted)
    redacted = re.sub(cc_pattern, "[CREDIT CARD REDACTED]", redacted)
    
    state['redacted_text'] = redacted
    
    if has_ssn or has_cc:
        state['leak_passed'] = False
        state['leak_message'] = f"Sensitive leaks redacted: {'SSN' if has_ssn else ''} {'Credit Card' if has_cc else ''}".strip()
        state['logs'].append("[check_leaks]: Redacted sensitive credential pattern match.")
    else:
        state['leak_passed'] = True
        state['leak_message'] = "Pass"
        state['logs'].append("[check_leaks]: Verified credential safety.")
    return state

def evaluate_overall(state: BasicState) -> BasicState:
    passed = state['keyword_passed'] and state['constraint_passed'] and state['leak_passed']
    state['overall_passed'] = passed
    state['logs'].append(f"[evaluate_overall]: Evaluation finished with status: {'PASSED' if passed else 'BLOCKED'}")
    return state

### 3. Compile the StateGraph

In [ ]:
builder = StateGraph(BasicState)
builder.add_node("keywords", check_keywords)
builder.add_node("constraints", check_constraints)
builder.add_node("leaks", check_leaks)
builder.add_node("evaluate", evaluate_overall)

builder.add_edge(START, "keywords")
builder.add_edge("keywords", "constraints")
builder.add_edge("constraints", "leaks")
builder.add_edge("leaks", "evaluate")
builder.add_edge("evaluate", END)

graph = builder.compile()
print("Basic Guardrails LangGraph compiled successfully!")

### 4. Run Test Cases

In [ ]:
def print_run(result):
    print("\n--- TRACE LOGS ---")
    for log in result['logs']:
        print(log)
    print("------------------")
    print("Overall Status:", "PASSED" if result['overall_passed'] else "BLOCKED")
    print("Redacted Output:", result['redacted_text'])
    print("Keyword Message:", result['keyword_message'])
    print("Constraint Message:", result['constraint_message'])
    print("Leak Message:", result['leak_message'])
    print("==================================\n")

# Case 1: Safe Input
print("=== Case 1: Safe Input ===")
res1 = graph.invoke({
    "text": "Hello, this is a friendly request to process some data.",
    "redacted_text": "", "keyword_passed": True, "keyword_message": "",
    "constraint_passed": True, "constraint_message": "",
    "leak_passed": True, "leak_message": "",
    "overall_passed": True, "logs": []
})
print_run(res1)

# Case 2: Prohibited Keyword
print("=== Case 2: Prohibited Keyword (sudo) ===")
res2 = graph.invoke({
    "text": "I need sudo access to execute root privileges.",
    "redacted_text": "", "keyword_passed": True, "keyword_message": "",
    "constraint_passed": True, "constraint_message": "",
    "leak_passed": True, "leak_message": "",
    "overall_passed": True, "logs": []
})
print_run(res2)

# Case 3: Length Bound Violation
print("=== Case 3: Length Violation (>100 characters) ===")
res3 = graph.invoke({
    "text": "This text is deliberately padded to exceed the one hundred character boundary limit. It has excessive words that will trip the length guardrail.",
    "redacted_text": "", "keyword_passed": True, "keyword_message": "",
    "constraint_passed": True, "constraint_message": "",
    "leak_passed": True, "leak_message": "",
    "overall_passed": True, "logs": []
})
print_run(res3)

# Case 4: Special Bracket Injection
print("=== Case 4: Bracket Injection (HTML) ===")
res4 = graph.invoke({
    "text": "Is it okay to use <html> tags here?",
    "redacted_text": "", "keyword_passed": True, "keyword_message": "",
    "constraint_passed": True, "constraint_message": "",
    "leak_passed": True, "leak_message": "",
    "overall_passed": True, "logs": []
})
print_run(res4)

# Case 5: Sensitive Data Leak
print("=== Case 5: Sensitive Leak (SSN & Credit Card) ===")
res5 = graph.invoke({
    "text": "My SSN is 123-45-6789 and my card number is 4111-2222-3333-4444.",
    "redacted_text": "", "keyword_passed": True, "keyword_message": "",
    "constraint_passed": True, "constraint_message": "",
    "leak_passed": True, "leak_message": "",
    "overall_passed": True, "logs": []
})
print_run(res5)